# SVM（1995）

- **核心原理**：最大間隔、Kernel Trick
- **變體**：soft margin（$C$）、各種 kernel（linear、polynomial、RBF）；SVR（回歸）、One-Class SVM（異常偵測）；`LinearSVC`（大資料的線性版）
- **推薦資料集**：Wine（178 支義大利葡萄酒、13 個化學成分、3 個品種）

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

SVM 把前幾章的好幾條線索接在一起：

- 跟**第一章感知器**一樣找一個分隔的超平面 $w \cdot x + b = 0$，但感知器「分開就好」，SVM 要找**離兩類最遠**（間隔最大）的那一個；
- 跟**第三章邏輯回歸**一樣是「loss + 正則化」，只是 loss 換成 **hinge loss**：分得夠開的點 loss 直接是 0，只有靠近邊界的少數點（**支持向量**）決定答案；
- 跟**第四章 KNN**一樣靠「相似度」做預測：kernel $K(x, z)$ 就是一種相似度，RBF kernel 直接用到歐氏距離 —— 所以**尺度**又變得很重要。

這一章的重點：**間隔是什麼、hinge loss 為什麼讓解變稀疏、對偶問題與 SMO 怎麼一次只動兩個變數、kernel trick 為什麼「不用算出高維特徵」、$C$ 與 $\gamma$ 怎麼控制複雜度。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、3C、3D、3E、4、5、7 節開頭另有各自的符號表或名詞表。

**資料**：每一列是一支葡萄酒，13 個欄位都是連續的化學量測（酒精濃度、蘋果酸、類黃酮 flavanoids、顏色強度 color_intensity、脯胺酸 proline……），**單位與量級差很多**（proline 約 300–1,500，hue 約 0.5–1.7）。目標是 3 個品種（0、1、2）。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $n$ | 訓練樣本數（二元子問題時是那兩類的樣本數） | 142（全部；以載入結果為準） |
| $d$ | 特徵數 | 13（3A、3E 的圖用 2 個） |
| $y_i$ | **二元**標籤，取值 $\pm 1$（不是 0／1！） | — |
| $w, b$ | 超平面的法向量與截距：$f(x) = w \cdot x + b$ | $(d,)$、純量 |
| $y_i f(x_i)$ | **函數間隔**（functional margin）：> 0 分對、≥ 1 「分得夠開」 | — |
| $2 / \lVert w\rVert$ | 兩條間隔線 $f = \pm 1$ 之間的寬度 | — |
| $C$ | soft margin 的懲罰係數：越大越不容忍錯誤 = 越複雜 | — |
| $\alpha_i$ | 對偶變數，$0 \le \alpha_i \le C$；$\alpha_i > 0$ 的點是**支持向量** | $(n,)$ |
| $K(x, z)$ | kernel：在某個（可能無限維的）特徵空間裡的內積 | — |
| $\gamma$ | RBF kernel 的寬度參數：$K = e^{-\gamma\lVert x - z\rVert^2}$，越大越「局部」 = 越複雜 | — |
| OvO | one-vs-one：$K$ 類兩兩訓練一個二元 SVM，共 $K(K-1)/2$ 個，投票 | 3 個 |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第一章感知器 | 第一部分第 1 節（`01-classic-ml/01-perceptron`） | 只要分錯就更新 $w \leftarrow w + y_i x_i$；可分時一定收斂，但停在「第一個」能分開的超平面，不是最好的那一個。 |
| 第三章 | 第一部分第 3 節：邏輯回歸 | 最小化 log loss $\log(1 + e^{-y f(x)})$（$y = \pm 1$ 的寫法）+ L2 正則化；$C$ 越大正則化越弱；過擬合在 $C$ 大的那一邊。 |
| 第三章「標準化是必要的」 | 同上 | 梯度下降時尺度差異讓收斂極慢；L2 懲罰對不同尺度的係數不公平。 |
| 第四章 | 第一部分第 4 節：KNN | 縮放 = 改變每個特徵在距離裡的權重；單位不同時必須標準化。距離矩陣用 $\lVert a\rVert^2 + \lVert b\rVert^2 - 2a\cdot b$ 一次算完（TODO 3）。 |
| 第四章的 1-SE | 同上，TODO 15 | 跟 CV 最佳差不到一個標準差的候選中挑最簡單的；但要看「簡單」換到什麼。 |
| 第五章 | 第一部分第 5 節：決策樹 | 單調轉換不影響樹；樹不需要標準化。 |
| 第六章 | 第一部分第 6 節：樸素貝氏 | 生成式 vs 判別式；SVM 是判別式，而且連機率都不學，只學邊界。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 內積與長度**：$a \cdot b = \sum_j a_j b_j$、$\lVert a\rVert = \sqrt{a \cdot a}$。例：$a = (3, 4)$ → $\lVert a\rVert = 5$。

**2. 點到超平面的距離**：超平面 $\{x : w \cdot x + b = 0\}$，點 $x_0$ 到它的距離是 $\dfrac{|w \cdot x_0 + b|}{\lVert w\rVert}$。例：$w = (3, 4)$、$b = -5$、$x_0 = (3, 4)$ → $|9 + 16 - 5| / 5 = 4$。
所以 $f(x) = \pm 1$ 兩條線到 $f = 0$ 的距離都是 $1 / \lVert w\rVert$，兩條線之間寬 $2 / \lVert w\rVert$ —— **$\lVert w\rVert$ 越小，間隔越寬**。

**3. $\max(0, z)$ 與次梯度**：$h(z) = \max(0, 1 - z)$ 在 $z < 1$ 時斜率 $-1$、$z > 1$ 時斜率 0，在 $z = 1$ 有個折角（不可微）。次梯度（subgradient）= 折角處取兩邊斜率之間的任何一個值（實作通常取 0）。

**4. 約束最佳化與拉格朗日乘數（只需要直覺）**：在限制 $g(x) \le 0$ 下最小化 $f(x)$，可以給每個限制一個乘數 $\alpha \ge 0$，轉成「對偶問題」。最重要的結論是**互補鬆弛（complementary slackness）**：最佳解上，每個限制要嘛「剛好卡住」（$g = 0$），要嘛乘數 $\alpha = 0$。
在 SVM 裡：限制是「每個點都要分得夠開 $y_i f(x_i) \ge 1$」，乘數就是 $\alpha_i$ → **沒有卡在間隔線上（或越界）的點，$\alpha_i = 0$，對答案毫無影響**。

**5. 二次函數在區間上的最小值**：$g(t) = \frac{1}{2} a t^2 + c t$（$a > 0$）的最小值在 $t^* = -c / a$；如果 $t$ 被限制在 $[L, H]$，答案就是把 $t^*$ 夾（clip）進區間：$\min(H, \max(L, t^*))$。（SMO 每一步就是解這個。）

**6. 特徵映射與 kernel**：把 $x$ 映射到新的特徵 $\phi(x)$，再在新空間做線性模型。如果演算法只用到內積 $\phi(x) \cdot \phi(z)$，就可以用一個函數 $K(x, z)$ 直接算，不必真的建出 $\phi$。例：$x = (x_1, x_2)$，$\phi(x) = (x_1^2, \sqrt{2} x_1 x_2, x_2^2)$ → $\phi(x) \cdot \phi(z) = (x \cdot z)^2$。

**7. RBF kernel**：$K(x, z) = \exp(-\gamma \lVert x - z\rVert^2)$ —— 距離 0 時是 1，越遠越接近 0；$\gamma$ 越大，「多遠算遠」的半徑越小（約 $1/\sqrt{\gamma}$）。

**8. 偏差—變異**（重述）：模型越彈性（$C$ 大、$\gamma$ 大），偏差越低、變異越高。

#### 程式

**9. 平方距離矩陣**（第四章 TODO 3 的重述）：

```python
D = (A**2).sum(axis=1)[:, None] + (B**2).sum(axis=1)[None, :] - 2 * A @ B.T   # (n, m)
D = np.maximum(D, 0)   # 浮點誤差可能出現 -1e-12
```

**10. 布林遮罩挑子集**：`m = np.isin(y, [1, 2])` → `X[m]`、`y[m]` 只留兩類；`np.where(y[m] == 2, 1.0, -1.0)` 把標籤換成 $\pm 1$。

**11. `np.clip(x, L, H)`**：夾進區間。`rng.integers(n - 1)` 再 `j += (j >= i)`：從 $0..n-1$ 中隨機挑一個「不是 $i$」的數。

**12. `itertools.combinations(range(3), 2)`** → `(0, 1), (0, 2), (1, 2)`：OvO 的所有類別對。

**13. `np.meshgrid` 畫決策邊界**：在二維平面上撒一張格點，每個格點都預測一次，再用 `contourf` 上色、`contour(levels=[-1, 0, 1])` 畫邊界與間隔線。

---
## 1. 資料與探索性資料分析（EDA）

觀察直接寫在每格程式最上面的註解（`# 觀察：`）。每格都附了「看什麼」的提示。

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_wine

X_train, X_test, y_train, y_test = load_wine()
y_train = pd.Series(np.asarray(y_train), index=X_train.index, name="target").astype(int)
y_test = pd.Series(np.asarray(y_test), index=X_test.index, name="target").astype(int)
train = X_train.assign(target=y_train)

print("train / test：", X_train.shape, X_test.shape)
display(X_train.head())

In [ ]:
# 型態、缺值、類別分布
# 觀察：（有沒有缺值？三類各多少？測試集只有幾筆 —— 1 筆佔多少 accuracy？）
print("缺值：", int(X_train.isna().sum().sum()))
display(
    pd.DataFrame({"train": y_train.value_counts().sort_index(), "test": y_test.value_counts().sort_index()})
    .assign(train_ratio=lambda d: (d["train"] / d["train"].sum()).round(3))
)
print(f"測試集 {len(y_test)} 筆：1 筆 = {1 / len(y_test):.1%} 的 accuracy")

In [ ]:
# 尺度：每個特徵的範圍
# 觀察：（最大和最小的 std 差幾倍？哪個特徵的數值最大？如果直接算兩支酒的歐氏距離，誰會主導？—— 對照第四章）
desc = X_train.describe().T[["mean", "std", "min", "max"]]
display(desc.round(2).sort_values("std", ascending=False))
d2 = ((X_train - X_train.mean()) ** 2).mean()  # 每個特徵對平方距離的平均貢獻 ∝ 變異數（第四章前置複習 3）
print("各特徵佔「原始」平方距離的比例（前 3 名）：")
display((d2 / d2.sum()).sort_values(ascending=False).head(3).round(4))

In [ ]:
# 每個特徵在三類的分布（先標準化，才能畫在同一張圖）
# 觀察：（哪幾個特徵單獨就能把某一類分出來？哪一類最容易跟別類混？）
z = (X_train - X_train.mean()) / X_train.std()
long = z.assign(target=y_train).melt(id_vars="target", var_name="feature", value_name="z")
fig, ax = plt.subplots(figsize=(14, 4))
sns.boxplot(data=long, x="feature", y="z", hue="target", ax=ax, fliersize=2)
ax.tick_params(axis="x", rotation=45)
ax.set_title("Standardized features by class")
plt.show()

# 二維散點：本章 3A、3E 畫圖用的兩個特徵
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (fx, fy) in zip(axes, [("flavanoids", "color_intensity"), ("proline", "od280/od315_of_diluted_wines")]):
    for k in range(3):
        m = y_train == k
        ax.scatter(X_train.loc[m, fx], X_train.loc[m, fy], s=18, label=f"class {k}")
    ax.set_xlabel(fx)
    ax.set_ylabel(fy)
    ax.legend()
plt.show()

In [ ]:
# 特徵之間的相關
# 觀察：（哪幾對特徵高度相關（|r| > 0.7）？對 SVM 這種「找一個超平面」的模型，相關的特徵是問題嗎？—— 想想第三章的共線性）
corr = X_train.corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, cmap="coolwarm", vmin=-1, vmax=1, annot=True, fmt=".1f", annot_kws={"size": 7}, ax=ax)
plt.show()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().sort_values(key=np.abs, ascending=False)
display(pairs.head(6).round(3).rename("r"))

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 尺度：
#     (a) 哪個特徵佔「原始」平方距離的比例最大？大約多少？
#     (b) RBF kernel 是 exp(−γ‖x − z‖²)。不標準化的話，RBF SVM 實際上主要在看哪個特徵？
#     (c) 先預測：線性 SVM（不用距離，只有 w·x + b）需要標準化嗎？（提示：目標函數裡有 ½‖w‖²，跟第三章的 L2 一樣）
#         （3E 的實驗 3 會驗證）
# 答：
#
# (2) 從散點圖看：flavanoids × color_intensity 這兩個特徵，三類分得開嗎？哪兩類之間比較擠？
#     如果只能畫「一條直線」分開兩類，哪一對類別做得到、哪一對做不到？
# 答：
#
# (3) 資料量：訓練集 142 筆、測試集 36 筆。測試集上「差 1 筆」是多少 accuracy？
#     這對「用測試集比較兩個模型」有什麼影響？調參時該怎麼讓 CV 的估計更穩定？（第 5 節會用 RepeatedStratifiedKFold）
# 答：

---
## 2. 前處理與 Baseline

除了第五章的決策樹，前幾章都需要標準化；SVM 也一樣（理由在 TODO 1 (1)、3E 實驗 3）。`StandardScaler` 一律包在 `Pipeline` 裡，CV 每個 fold 只用那個 fold 的訓練部分算平均與標準差。

#### Baseline

- **Dummy prior**：永遠猜最多的那一類，機率給類別比例。
- **NearestCentroid（標準化後）**：每一類一個平均點，找最近的中心（第四章的 baseline）—— 邊界是兩兩中心的垂直平分線，也是一種線性分類器。

#### 本節名詞

- **accuracy**：三類大致平衡，用 accuracy 當主要指標。
- **`RepeatedStratifiedKFold(n_splits=5, n_repeats=10)`**：5-fold 分層 CV 重複 10 次（每次重新打亂），共 50 個分數。資料很少時，單次 5-fold 的結果會隨切法跳動，重複可以讓平均比較穩。

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_score, cross_validate
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

# 整章共用的 CV：5-fold 分層、重複 10 次
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
cv_single = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 畫 validation curve 等比較慢的地方用


def cv_summary(model, X: pd.DataFrame = X_train, y: pd.Series = y_train) -> dict[str, str]:
    """重複 5-fold CV 的 accuracy「平均 ± 標準差」，以及最差、最好的 fold。"""
    s = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    return {"accuracy": f"{s.mean():.4f} ± {s.std():.4f}", "worst fold": f"{s.min():.3f}", "best fold": f"{s.max():.3f}"}


baseline = pd.DataFrame(
    {
        "Dummy prior": cv_summary(DummyClassifier(strategy="prior")),
        "NearestCentroid (raw)": cv_summary(NearestCentroid()),
        "NearestCentroid (standardized)": cv_summary(make_pipeline(StandardScaler(), NearestCentroid())),
    }
).T
display(baseline)

In [ ]:
# TODO 2（問答）
#
# (1) NearestCentroid 標準化前後差多少？用 TODO 1 (1) 的結論解釋。
# 答：
#
# (2) 標準化的 NearestCentroid 已經很高了。這代表 Wine 這份資料「三類大致是三團、彼此離得不近」。
#     那 SVM 還能進步的空間在哪裡？（提示：看 worst fold —— 錯的是哪些點？中心法只用「平均」，SVM 用的是哪些點？）
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 線性 SVM 的**原始問題**（primal）：hinge loss + 次梯度下降 | 3 |
| 3B | kernel：RBF kernel 矩陣；kernel trick 為什麼成立 | 5 |
| 3C | **對偶問題**（dual）與簡化版 SMO：一次只動兩個 $\alpha$ | 6、7、8 |
| 3D | 多類別：one-vs-one 投票 | 10 |
| 3E | 實驗：$C$、$\gamma$、尺度、hinge vs log loss | — |

**二元子問題**：3A–3C 都是二元分類，標籤用 $\pm 1$。3A 用「類別 1 vs 類別 2」、只取 2 個特徵（flavanoids、color_intensity，標準化後），方便畫圖；3C 用全部 13 個特徵、三對類別各做一次。

### 3A. 線性 SVM：hinge loss

**最大間隔**：能分開兩類的直線有無限多條（感知器停在第一條），SVM 選**離最近的點最遠**的那一條。把「最近的點」定成 $y_i f(x_i) = 1$，間隔寬度就是 $2 / \lVert w\rVert$（前置複習 2）—— 最大化間隔 = 最小化 $\frac{1}{2}\lVert w\rVert^2$，限制每個點 $y_i(w \cdot x_i + b) \ge 1$（hard margin）。

**soft margin**：資料分不開（或有雜訊）時，允許點越界，但每越界一點要付錢：

$$J(w, b) = \frac{1}{2}\lVert w\rVert^2 + C \sum_{i=1}^{n} \max\big(0,\ 1 - y_i (w \cdot x_i + b)\big)$$

- 第一項：正則化（間隔寬）；第二項：**hinge loss**（$y_i f(x_i) \ge 1$ 的點 loss = 0）。
- $C$ 大：越界很貴 → 間隔窄、盡量分對（複雜）；$C$ 小：間隔寬、容忍錯誤（簡單）。$C \to \infty$ 就是 hard margin。
- 跟第三章比：邏輯回歸是 $\frac{1}{2}\lVert w\rVert^2 + C\sum \log(1 + e^{-y_i f(x_i)})$ —— 只是把 hinge 換成 log loss。

**次梯度**（前置複習 3）：令 $m_i = y_i(w \cdot x_i + b)$，「active」的點 = $m_i < 1$（越界或在間隔內）：

$$\frac{\partial J}{\partial w} = w - C \sum_{i:\ m_i < 1} y_i x_i, \qquad \frac{\partial J}{\partial b} = -C \sum_{i:\ m_i < 1} y_i$$

只有 active 的點會「拉」超平面；分得夠開的點完全不影響梯度。用遞減的學習率 $\eta_t = \eta_0 / \sqrt{t}$ 做（次）梯度下降。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $X$ | 標準化後的特徵（二元子集） | $(n, d)$ | `X` |
| $y$ | $\pm 1$ 標籤 | $(n,)$ | `y` |
| $w$、$b$ | 參數 | $(d,)$、純量 | `self.w_`、`self.b_` |
| $m$ | 函數間隔 $y_i(w\cdot x_i + b)$ | $(n,)$ | `margins` |
| active | $m_i < 1$ 的點 | $(n,)$ bool | `active` |
| $\partial J/\partial w$、$\partial J/\partial b$ | 次梯度（是結果，用來更新參數） | $(d,)$、純量 | `grad_w`、`grad_b` |

**下標公式**（TODO 3）：$\frac{\partial J}{\partial w_j} = w_j - C\sum_i [m_i < 1]\, y_i X_{ij}$ —— 加總 $i$、保留 $j$。可以寫成 `(active * y) @ X`：`active * y` 是 $(n,)$（不是 active 的位置變 0），再跟 $(n, d)$ 的 $X$ 做矩陣乘法 → $(d,)$。

#### 最小例子：TODO 3（hinge loss 與次梯度）

3 個點、2 個特徵：$X = \{(2, 0), (0.5, 0), (-1, 0)\}$、$y = (+1, +1, -1)$；目前 $w = (1, 0)$、$b = 0$、$C = 1$。

1. **正確答案**：逐點算 $m_i$、hinge、是否 active，再累加次梯度。目標：$m = (2, 0.5, 1)$、hinge $= (0, 0.5, 0)$、$J = 0.5 + 0.5 = 1$、$\partial J/\partial w = (1, 0) - (0.5, 0) = (0.5, 0)$、$\partial J/\partial b = -1$。
2. **積木**：`y * (X @ w + b)`、`np.maximum(0, 1 - m)`、`active * y` 的值、`(active * y) @ X`。
3. **留給你**：組成 $J$ 與兩個次梯度。

In [ ]:
# 最小例子：TODO 3（hinge loss 與次梯度）
X_mini = np.array([[2.0, 0.0], [0.5, 0.0], [-1.0, 0.0]])
y_mini = np.array([1.0, 1.0, -1.0])
w_mini, b_mini, C_mini = np.array([1.0, 0.0]), 0.0, 1.0

# ---- 1. 正確答案：逐點 ----
gw_loop, gb_loop, loss_loop = w_mini.copy(), 0.0, 0.5 * w_mini @ w_mini
for xi, yi in zip(X_mini, y_mini):
    mi = yi * (xi @ w_mini + b_mini)
    loss_loop += C_mini * max(0.0, 1 - mi)
    if mi < 1:
        gw_loop -= C_mini * yi * xi
        gb_loop -= C_mini * yi
    print(f"x={xi} y={yi:+.0f}  m={mi:.2f}  hinge={max(0.0, 1 - mi):.2f}  active={mi < 1}")
print("正確答案：J =", loss_loop, " grad_w =", gw_loop, " grad_b =", gb_loop)

# ---- 2. 積木 ----
m_vec = y_mini * (X_mini @ w_mini + b_mini)
print("\nmargins = y * (X @ w + b)", m_vec)
print("hinge = np.maximum(0, 1 − m)", np.maximum(0, 1 - m_vec))
active_mini = m_vec < 1
print("active * y               ", active_mini * y_mini, "← 不是 active 的點變 0")
print("(active * y) @ X         ", (active_mini * y_mini) @ X_mini, "← = Σ_{active} y_i x_i，shape (d,)")
print("注意：m 剛好 = 1 的點（第 3 個）不算 active —— 折角處次梯度取 0")

# ---- 3. 留給你 ----
# 問題：J = ½‖w‖² + C·Σ hinge；grad_w = w − C·(哪個積木)；grad_b = −C·(哪個積木的總和)？
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(LinearSVMPrimal(C=1.0).objective_and_grad(X_mini, y_mini, w_mini, b_mini))

In [ ]:
from typing import Self

from numpy.typing import ArrayLike

FEAT2 = ["flavanoids", "color_intensity"]


def binary_subset(X: pd.DataFrame | np.ndarray, y: ArrayLike, pos: int, neg: int) -> tuple[np.ndarray, np.ndarray]:
    """只留 pos、neg 兩類；回傳 (X 子集, ±1 標籤)，pos → +1、neg → −1。"""
    y = np.asarray(y)
    m = np.isin(y, [pos, neg])
    return np.asarray(X, dtype=float)[m], np.where(y[m] == pos, 1.0, -1.0)


def plot_svm_2d(ax, decision, X: np.ndarray, y: np.ndarray, title: str, sv: np.ndarray | None = None) -> None:
    """二維決策邊界：f = 0 實線、f = ±1 虛線；sv 是支持向量的布林遮罩（畫空心圈）。"""
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300), np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))
    F = decision(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, F, levels=[-1e9, 0, 1e9], colors=["#d6e4f0", "#f6d9cc"], alpha=0.8)
    ax.contour(xx, yy, F, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"], linewidths=1)
    ax.scatter(X[y < 0, 0], X[y < 0, 1], s=15, c="C0", label="−1")
    ax.scatter(X[y > 0, 0], X[y > 0, 1], s=15, c="C1", label="+1")
    if sv is not None:
        ax.scatter(X[sv, 0], X[sv, 1], s=80, facecolors="none", edgecolors="k", label="support vectors")
    ax.set_title(title)
    ax.set_xlabel(FEAT2[0] + " (std)")
    ax.set_ylabel(FEAT2[1] + " (std)")


class LinearSVMPrimal:
    """線性 soft-margin SVM，用次梯度下降最小化 ½‖w‖² + C Σ hinge。標籤必須是 ±1。"""

    def __init__(self, C: float = 1.0, lr0: float = 0.05, n_iter: int = 20000) -> None:
        self.C = C
        self.lr0 = lr0
        self.n_iter = n_iter

    def objective_and_grad(self, X: np.ndarray, y: np.ndarray, w: np.ndarray, b: float) -> tuple[float, np.ndarray, float]:
        """回傳 (J, ∂J/∂w (d,), ∂J/∂b)。"""
        margins = y * (X @ w + b)  # (n,)
        active = margins < 1  # (n,) bool
        # TODO 3（程式）：目標函數與次梯度
        #   - J = ½ w·w + C · Σ max(0, 1 − margins)
        #   - grad_w = w − C · (active * y) @ X        （只有 active 的點有貢獻）
        #   - grad_b = −C · Σ (active * y)
        raise NotImplementedError("TODO 3：hinge 與次梯度")
        return float(J), grad_w, float(grad_b)

    def fit(self, X: ArrayLike, y: ArrayLike) -> Self:
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        w, b = np.zeros(X.shape[1]), 0.0
        self.history_ = []
        for t in range(1, self.n_iter + 1):
            J, gw, gb = self.objective_and_grad(X, y, w, b)
            self.history_.append(J)
            lr = self.lr0 / np.sqrt(t)  # 遞減的學習率：次梯度法不保證每一步都下降
            w, b = w - lr * gw, b - lr * gb
        self.w_, self.b_ = w, b
        return self

    def decision_function(self, X: ArrayLike) -> np.ndarray:
        return np.asarray(X, dtype=float) @ self.w_ + self.b_

    def predict(self, X: ArrayLike) -> np.ndarray:
        return np.where(self.decision_function(X) >= 0, 1.0, -1.0)

In [ ]:
from sklearn.svm import SVC

sc2 = StandardScaler().fit(X_train[FEAT2])
X2, y2 = binary_subset(sc2.transform(X_train[FEAT2]), y_train, pos=2, neg=1)  # 類別 2 = +1、類別 1 = −1
primal = LinearSVMPrimal(C=1.0).fit(X2, y2)
sk_lin2 = SVC(kernel="linear", C=1.0).fit(X2, y2)
J_mine = primal.objective_and_grad(X2, y2, primal.w_, primal.b_)[0]
J_sk = primal.objective_and_grad(X2, y2, sk_lin2.coef_[0], sk_lin2.intercept_[0])[0]
print("手寫 w, b：", primal.w_.round(4), round(primal.b_, 4))
print("sklearn  ：", sk_lin2.coef_[0].round(4), sk_lin2.intercept_.round(4))
print(f"✅ 目標函數 J：手寫 {J_mine:.5f} vs sklearn {J_sk:.5f}（相對差應 < 1e-3）：{abs(J_mine - J_sk) / J_sk:.1e}")
print("✅ 預測一致比例：", np.mean(primal.predict(X2) == sk_lin2.predict(X2)), "（應 ≈ 1.0；剛好貼在邊界上的點可能不同）")


margins2 = y2 * primal.decision_function(X2)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_svm_2d(axes[0], primal.decision_function, X2, y2, "Primal (subgradient), C = 1", sv=margins2 <= 1 + 1e-3)
axes[0].legend(loc="upper left", fontsize=8)
axes[1].plot(primal.history_)
axes[1].set_yscale("log")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("J(w, b)")
axes[1].set_title("Objective during subgradient descent")
plt.show()
print(f"間隔寬度 2/‖w‖ = {2 / np.linalg.norm(primal.w_):.3f}；y·f(x) ≤ 1 的點：{int((margins2 <= 1 + 1e-3).sum())} / {len(y2)}")

In [ ]:
# TODO 4（問答）間隔與 hinge loss
#
# (1) 看左圖：
#     (a) 實線、兩條虛線各是 f(x) = 多少？間隔寬度是多少？
#     (b) 空心圈的點有什麼共同點？它們以外的點，如果往遠離邊界的方向移動一點點，w、b 會不會變？為什麼？
#         （提示：3A 的次梯度只有 active 的點有貢獻）
#     (c) 這兩類在這兩個特徵上可以完全分開，但 C = 1 的解仍然有點落在間隔裡。這是 soft margin 還是 hard margin 的行為？
# 答：
#
# (2) 右圖：J 一路下降嗎？為什麼次梯度法需要「遞減」的學習率？（提示：折角處的次梯度會在兩邊來回跳）
# 答：
#
# (3) 對照第一章感知器：感知器的更新只在「分錯」（y·f < 0）時發生，SVM 的 active 條件是 y·f < 1。
#     多出來的「0 ≤ y·f < 1」這段，代表 SVM 在要求什麼？這為什麼會讓解比較「穩」？
# 答：

### 3B. Kernel

**對偶問題**（3C 會用）的形式裡，資料只以**內積** $x_i \cdot x_j$ 的形式出現。把內積換成 kernel $K(x_i, x_j)$，就等於先把資料映射到 $\phi(x)$、再在新空間做線性 SVM —— 卻**不用真的算出** $\phi(x)$（前置複習 6）。

| kernel | $K(x, z)$ | 對應的特徵空間 | sklearn |
| --- | --- | --- | --- |
| linear | $x \cdot z$ | 原空間 | `kernel="linear"` |
| polynomial | $(\gamma\, x \cdot z + r)^p$ | 所有 $\le p$ 次的單項式（$d = 13$、$p = 3$：560 維） | `kernel="poly"` |
| RBF | $\exp(-\gamma\lVert x - z\rVert^2)$ | **無限維** | `kernel="rbf"`（預設） |

`gamma="scale"`（sklearn 預設）$= 1 / (d \cdot \text{Var}(X))$；標準化後 $\text{Var}(X) = 1$，就是 $1/d$。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $A$、$B$ | 兩組樣本 | $(n, d)$、$(m, d)$ | `A`、`B` |
| $D_{ij}$ | 平方距離 $\lVert a_i - b_j\rVert^2$ | $(n, m)$ | `D` |
| $K_{ij}$ | kernel 值 | $(n, m)$ | 回傳值 |
| Gram 矩陣 | 訓練集對自己的 kernel 矩陣 $K(X, X)$ | $(n, n)$ | `K` |

**下標公式**（TODO 5）：$K_{ij} = \exp(-\gamma D_{ij})$，$D_{ij} = \lVert a_i\rVert^2 + \lVert b_j\rVert^2 - 2\,a_i\cdot b_j$ —— 跟第四章 TODO 3 的距離矩陣一樣，只是外面多套一個 $\exp(-\gamma\,\cdot)$。

#### 最小例子：TODO 5（RBF kernel 矩陣；kernel trick）

1. **正確答案**：兩層迴圈逐對算 $\exp(-\gamma\lVert a_i - b_j\rVert^2)$。另外驗證 kernel trick：$\phi(x) = (x_1^2, \sqrt{2}x_1x_2, x_2^2)$ 的內積 = $(x \cdot z)^2$。
2. **積木**：平方距離矩陣（第四章）、`np.exp(-gamma * D)`、對角線的值。
3. **留給你**：把兩個積木接起來。

In [ ]:
# 最小例子：TODO 5（RBF kernel 矩陣；kernel trick）
A_mini = np.array([[0.0, 0.0], [1.0, 1.0]])
B_mini = np.array([[0.0, 0.0], [1.0, 0.0], [3.0, 4.0]])
gamma_mini = 0.5

# ---- 1. 正確答案：迴圈 ----
K_loop = np.array([[np.exp(-gamma_mini * np.sum((a - b) ** 2)) for b in B_mini] for a in A_mini])
print("正確答案 K =\n", K_loop.round(4))

phi = lambda v: np.array([v[0] ** 2, np.sqrt(2) * v[0] * v[1], v[1] ** 2])
x_k, z_k = np.array([1.0, 2.0]), np.array([3.0, -1.0])
print(f"\nkernel trick：φ(x)·φ(z) = {phi(x_k) @ phi(z_k):.4f}，(x·z)² = {(x_k @ z_k) ** 2:.4f} ← 一樣，但右邊不用建出 φ")

# ---- 2. 積木 ----
D_mini = np.maximum((A_mini**2).sum(axis=1)[:, None] + (B_mini**2).sum(axis=1)[None, :] - 2 * A_mini @ B_mini.T, 0)
print("\n平方距離 D =\n", D_mini)
print("點跟自己的 kernel 值 exp(−γ·0) =", np.exp(-gamma_mini * 0.0), "← RBF 的 Gram 矩陣對角線一定是 1")

# ---- 3. 留給你 ----
# 問題：K = ?（用 D 與 gamma）
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(rbf_kernel_matrix(A_mini, B_mini, gamma_mini).round(4))

In [ ]:
from collections.abc import Callable


def linear_kernel_matrix(A: ArrayLike, B: ArrayLike, gamma: float | None = None) -> np.ndarray:
    """K[i, j] = A[i] · B[j]；(n, d)、(m, d) → (n, m)。gamma 不用，只是讓介面跟 RBF 一樣。"""
    return np.asarray(A, dtype=float) @ np.asarray(B, dtype=float).T


def rbf_kernel_matrix(A: ArrayLike, B: ArrayLike, gamma: float) -> np.ndarray:
    """K[i, j] = exp(−γ ‖A[i] − B[j]‖²)；(n, d)、(m, d) → (n, m)。"""
    A, B = np.asarray(A, dtype=float), np.asarray(B, dtype=float)
    # TODO 5（程式）：RBF kernel 矩陣
    #   - 平方距離 D 用 ‖a‖² + ‖b‖² − 2a·b（第四章 TODO 3），np.maximum(D, 0) 修掉浮點誤差
    #   - 不要用 A[:, None, :] − B[None, :, :]（n×m×d 的暫存陣列）
    raise NotImplementedError("TODO 5：RBF kernel")


from sklearn.metrics.pairwise import rbf_kernel

sc = StandardScaler().fit(X_train)
Xtr, Xte = sc.transform(X_train), sc.transform(X_test)  # 標準化後的全部 13 個特徵（3C 之後共用）
ytr, yte = y_train.to_numpy(), y_test.to_numpy()
GAMMA = 1 / Xtr.shape[1]  # 標準化後 gamma="scale" = 1/d
K_check = rbf_kernel_matrix(Xtr, Xtr, GAMMA)
print("✅ 與 sklearn rbf_kernel 的最大差異：", np.abs(K_check - rbf_kernel(Xtr, Xtr, gamma=GAMMA)).max(), "（應 < 1e-12）")
print("✅ 對角線都是 1：", np.allclose(np.diag(K_check), 1), "；對稱：", np.allclose(K_check, K_check.T))
print("✅ 半正定（最小特徵值 ≥ −1e-10）：", np.linalg.eigvalsh(K_check).min() >= -1e-10)

### 3C. 對偶問題與簡化版 SMO

**對偶問題**（由前置複習 4 的拉格朗日乘數推出來；推導放待補強）：

$$\max_{\alpha}\ W(\alpha) = \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j K(x_i, x_j) \quad \text{s.t.}\ \ 0 \le \alpha_i \le C,\ \ \sum_i \alpha_i y_i = 0$$

解出 $\alpha$ 之後，**決策函數**是

$$f(x) = \sum_{i} \alpha_i y_i K(x_i, x) + b$$

—— 一個「以支持向量為中心、用 kernel 相似度加權投票」的模型（對照第四章 KNN：KNN 用所有訓練點、每人一票；SVM 只用 $\alpha_i > 0$ 的點、票數 $\alpha_i y_i$ 是學出來的）。

**KKT 條件**（前置複習 4 的互補鬆弛，最佳解上每個點都滿足其中一個）：

| $\alpha_i$ | 對應 | $y_i f(x_i)$ |
| --- | --- | --- |
| $= 0$ | 分得夠開，不是支持向量 | $\ge 1$ |
| $0 < \alpha_i < C$ | 剛好在間隔線上（free support vector） | $= 1$ |
| $= C$ | 在間隔內或分錯（bound support vector） | $\le 1$ |

**SMO（Sequential Minimal Optimization，Platt 1998）**：限制 $\sum_i \alpha_i y_i = 0$ 讓我們不能只改一個 $\alpha$（改一個就破壞等式），所以**每次挑兩個** $\alpha_i, \alpha_j$，固定其他所有 $\alpha$，這時 $W$ 變成一個變數的二次函數，有閉式解（前置複習 5）：

1. 誤差 $E_k = f(x_k) - y_k$（TODO 6）。
2. $\alpha_j$ 的可行區間 $[L, H]$（TODO 7）：
   - $y_i \ne y_j$：$L = \max(0, \alpha_j - \alpha_i)$、$H = \min(C, C + \alpha_j - \alpha_i)$
   - $y_i = y_j$：$L = \max(0, \alpha_i + \alpha_j - C)$、$H = \min(C, \alpha_i + \alpha_j)$
   （由 $\alpha_i y_i + \alpha_j y_j$ 必須維持不變、兩者都在 $[0, C]$ 推出來。）
3. $\eta = 2K_{ij} - K_{ii} - K_{jj}$（$\le 0$，是二次函數的曲率）；
   $\alpha_j^\text{new} = \text{clip}\big(\alpha_j - y_j (E_i - E_j)/\eta,\ L,\ H\big)$；
   $\alpha_i^\text{new} = \alpha_i + y_i y_j(\alpha_j - \alpha_j^\text{new})$（TODO 8）。
4. 更新 $b$（template 已寫好）。

**簡化版**（Stanford CS229 的版本）：外層迴圈走過每個 $i$，只挑「違反 KKT 超過 `tol`」的 $i$，$j$ 隨機挑；連續 `max_passes` 輪都沒有任何 $\alpha$ 改變就停。因為 $j$ 是隨機挑的，「一輪沒進展」不代表已經到最佳解（可能只是運氣不好），所以 `max_passes` 不能太小（這裡用 30）。真正的 libsvm 用比較聰明的方法挑 $(i, j)$，收斂快很多，但會到同一個最佳解（對偶問題是凸的）。

#### 符號表（3C）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $K$ | 訓練集的 Gram 矩陣 | $(n, n)$ | `K` |
| $\alpha$ | 對偶變數 | $(n,)$ | `alpha`（fit 後 `self.alpha_`） |
| $\alpha \odot y$ | 每個點的「票數」$\alpha_i y_i$ | $(n,)$ | `alpha * y` |
| $f(x_k)$ | 第 $k$ 個訓練點的決策值 $= \sum_i \alpha_i y_i K_{ik} + b$ | 純量 | — |
| $E_k$ | 誤差 $f(x_k) - y_k$ | 純量 | `E_i`、`E_j` |
| $L$、$H$ | $\alpha_j$ 的可行區間 | 純量 | `L`、`H` |
| $\eta$ | 曲率 $2K_{ij} - K_{ii} - K_{jj}$ | 純量 | `eta` |
| 支持向量 | $\alpha_i > 10^{-6}$ 的點 | — | `self.support_` |

**下標公式**（TODO 6）：$f(x_k) = \sum_i (\alpha_i y_i) K_{ik} + b$ —— 加總 $i$ → `(alpha * y) @ K[:, k]`（一個長度 $n$ 的內積）。

#### 最小例子：TODO 6、7、8（SMO 的一步）

4 個一維的點：$x = (-2, -1, 1, 2)$、$y = (-1, -1, +1, +1)$，linear kernel，$C = 10$。從 $\alpha = 0$、$b = 0$ 開始，挑 $(i, j) = (1, 2)$（最靠近中間的兩個點）做一步。

1. **正確答案**：手算。$f \equiv 0$ → $E_1 = 0 - (-1) = 1$、$E_2 = 0 - 1 = -1$；$y_1 \ne y_2$ → $L = 0$、$H = 10$；$\eta = 2(-1) - 1 - 1 = -4$；$\alpha_2 = 0 - 1 \cdot (1 - (-1))/(-4) = 0.5$；$\alpha_1 = 0 + (-1)(1)(0 - 0.5) = 0.5$。這一步之後 $w = \sum \alpha_i y_i x_i = 0.5 \cdot 1 + 0.5 \cdot 1 = 1$，正好是最大間隔解（間隔線 $f = \pm 1$ 穿過 $x = \pm 1$）。
2. **積木**：`(alpha * y) @ K[:, k]`、兩種 $y_i, y_j$ 情況、`np.clip`。
3. **留給你**：寫進 `_error`、`_bounds`、`_update_pair`。

In [ ]:
# 最小例子：TODO 6、7、8（SMO 的一步）
x_smo = np.array([[-2.0], [-1.0], [1.0], [2.0]])
y_smo = np.array([-1.0, -1.0, 1.0, 1.0])
K_smo = x_smo @ x_smo.T
alpha_smo, b_smo, C_smo = np.zeros(4), 0.0, 10.0
i_s, j_s = 1, 2

# ---- 1. 正確答案：照公式一步一步 ----
f_loop = [sum(alpha_smo[t] * y_smo[t] * K_smo[t, k] for t in range(4)) + b_smo for k in range(4)]
E_i_loop, E_j_loop = f_loop[i_s] - y_smo[i_s], f_loop[j_s] - y_smo[j_s]
L_loop, H_loop = max(0, alpha_smo[j_s] - alpha_smo[i_s]), min(C_smo, C_smo + alpha_smo[j_s] - alpha_smo[i_s])  # y_i ≠ y_j
eta_loop = 2 * K_smo[i_s, j_s] - K_smo[i_s, i_s] - K_smo[j_s, j_s]
aj_loop = float(np.clip(alpha_smo[j_s] - y_smo[j_s] * (E_i_loop - E_j_loop) / eta_loop, L_loop, H_loop))
ai_loop = alpha_smo[i_s] + y_smo[i_s] * y_smo[j_s] * (alpha_smo[j_s] - aj_loop)
print(f"E_i = {E_i_loop}, E_j = {E_j_loop}, L = {L_loop}, H = {H_loop}, η = {eta_loop}")
print(f"正確答案：α_j = {aj_loop}, α_i = {ai_loop}")

# ---- 2. 積木 ----
print("\nalpha * y          ", alpha_smo * y_smo, "← 一開始全是 0，所以 f ≡ b = 0")
print("K[:, i]            ", K_smo[:, i_s], "← 第 i 個點跟每個訓練點的 kernel 值")
print("(alpha * y) @ K[:, i] + b =", (alpha_smo * y_smo) @ K_smo[:, i_s] + b_smo)
print("np.clip(1.7, 0, 1) =", np.clip(1.7, 0, 1), "；np.clip(-0.3, 0, 1) =", np.clip(-0.3, 0, 1))

# ---- 3. 留給你 ----
# 問題：_error 用哪個積木？_bounds 的兩種情況各是什麼？_update_pair 先更新 α_j（clip），再用「α_i y_i + α_j y_j 不變」更新 α_i。
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（一步之後應該是 α = [0, 0.5, 0.5, 0]）：
# s_mini = BinarySVM(C=10, kernel="linear"); s_mini.K_, s_mini.y_ = K_smo, y_smo; print(s_mini._update_pair(np.zeros(4), 0.0, 1, 2))

#### 程式結構圖：`BinarySVM`

```
BinarySVM(C=1.0, kernel="rbf", gamma=1/13, tol=1e-4, max_passes=30, seed=0)
├── 超參數：C、kernel、gamma、tol、max_passes、seed
├── fit(X, y)                       y 必須是 ±1
│   ├── 存下 X_ (n, d)、y_ (n,)、K_ = kernel(X_, X_) (n, n)
│   └── 主迴圈：for i in range(n)
│         ├── E_i = _error(alpha, b, i)                       ← TODO 6
│         ├── 違反 KKT？否 → 跳過
│         ├── 隨機挑 j ≠ i，E_j = _error(alpha, b, j)
│         └── alpha, b, changed = _update_pair(alpha, b, i, j)
│               ├── L, H = _bounds(alpha, i, j)               ← TODO 7
│               ├── η、α_j、α_i 的更新                         ← TODO 8
│               └── b 的更新（已寫好）
│       結束後存 alpha_ (n,)、b_、support_（α > 1e-6 的列號）、n_passes_
├── decision_function(X)  Σ_{支持向量} α_i y_i K(x_i, x) + b    → (n_query,)
└── predict(X)            sign(decision_function) → ±1
```

`_error`、`_bounds`、`_update_pair` 都直接讀 `self.K_`、`self.y_`；`alpha`、`b` 則當參數傳進傳出（方便最小例子單獨測一步）。

In [ ]:
class BinarySVM:
    """二元 soft-margin SVM，用簡化版 SMO 解對偶問題。標籤必須是 ±1。"""

    def __init__(self, C: float = 1.0, kernel: str = "rbf", gamma: float = 1 / 13, tol: float = 1e-4, max_passes: int = 30, seed: int = 0) -> None:
        self.C = C
        self.kernel = kernel
        self.gamma = gamma
        self.tol = tol
        self.max_passes = max_passes
        self.seed = seed

    def _kernel(self, A: np.ndarray, B: np.ndarray) -> np.ndarray:
        return linear_kernel_matrix(A, B) if self.kernel == "linear" else rbf_kernel_matrix(A, B, self.gamma)

    def _error(self, alpha: np.ndarray, b: float, k: int) -> float:
        """E_k = f(x_k) − y_k，f(x_k) = Σ_i α_i y_i K[i, k] + b。"""
        # TODO 6（程式）：第 k 個訓練點的誤差
        #   - (alpha * self.y_) 是每個點的票數 (n,)；self.K_[:, k] 是每個點跟第 k 個點的 kernel 值 (n,)
        raise NotImplementedError("TODO 6：E_k")

    def _bounds(self, alpha: np.ndarray, i: int, j: int) -> tuple[float, float]:
        """α_j 的可行區間 [L, H]。"""
        C, ai, aj = self.C, alpha[i], alpha[j]
        # TODO 7（程式）：L、H
        #   - y_i ≠ y_j：L = max(0, α_j − α_i)、H = min(C, C + α_j − α_i)
        #   - y_i = y_j：L = max(0, α_i + α_j − C)、H = min(C, α_i + α_j)
        raise NotImplementedError("TODO 7：L、H")

    def _update_pair(self, alpha: np.ndarray, b: float, i: int, j: int) -> tuple[np.ndarray, float, bool]:
        """對 (α_i, α_j) 做一步 SMO。回傳 (新的 alpha（複本）, 新的 b, 有沒有改變)。"""
        K, y = self.K_, self.y_
        E_i, E_j = self._error(alpha, b, i), self._error(alpha, b, j)
        L, H = self._bounds(alpha, i, j)
        eta = 2 * K[i, j] - K[i, i] - K[j, j]
        if L == H or eta >= 0:  # 沒有移動空間，或曲率不對（重複的點等特殊情況）
            return alpha, b, False
        a = alpha.copy()
        # TODO 8（程式）：更新 α_j、α_i
        #   - a[j] = clip(α_j − y_j (E_i − E_j) / η, L, H)
        #   - a[i] = α_i + y_i y_j (α_j − a[j])        （α_j 是「舊的」alpha[j]）
        raise NotImplementedError("TODO 8：α 的更新")
        if abs(a[j] - alpha[j]) < 1e-8:
            return alpha, b, False
        # b 的更新：讓 E_i 或 E_j（落在 0 < α < C 的那個）變成 0
        di, dj = a[i] - alpha[i], a[j] - alpha[j]
        b1 = b - E_i - y[i] * di * K[i, i] - y[j] * dj * K[i, j]
        b2 = b - E_j - y[i] * di * K[i, j] - y[j] * dj * K[j, j]
        if 0 < a[i] < self.C:
            b_new = b1
        elif 0 < a[j] < self.C:
            b_new = b2
        else:
            b_new = (b1 + b2) / 2
        return a, float(b_new), True

    def fit(self, X: ArrayLike, y: ArrayLike) -> Self:
        self.X_, self.y_ = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        self.K_ = self._kernel(self.X_, self.X_)
        n = len(self.y_)
        alpha, b = np.zeros(n), 0.0
        rng = np.random.default_rng(self.seed)
        passes = n_iter = 0
        while passes < self.max_passes:
            n_changed = 0
            n_iter += 1
            for i in range(n):
                E_i = self._error(alpha, b, i)
                yE = self.y_[i] * E_i  # = y_i f(x_i) − 1
                if (yE < -self.tol and alpha[i] < self.C) or (yE > self.tol and alpha[i] > 0):  # 違反 KKT
                    j = int(rng.integers(n - 1))
                    j += j >= i
                    alpha, b, changed = self._update_pair(alpha, b, i, j)
                    n_changed += changed
            passes = passes + 1 if n_changed == 0 else 0
        self.alpha_, self.b_, self.n_iter_ = alpha, b, n_iter
        self.support_ = np.flatnonzero(alpha > 1e-6)
        return self

    def dual_objective(self, alpha: np.ndarray | None = None) -> float:
        """W(α) = Σα − ½ (α⊙y)ᵀ K (α⊙y)。"""
        a = self.alpha_ if alpha is None else alpha
        v = a * self.y_
        return float(a.sum() - 0.5 * v @ self.K_ @ v)

    def decision_function(self, X: ArrayLike) -> np.ndarray:
        s = self.support_
        return self._kernel(np.asarray(X, dtype=float), self.X_[s]) @ (self.alpha_[s] * self.y_[s]) + self.b_

    def predict(self, X: ArrayLike) -> np.ndarray:
        return np.where(self.decision_function(X) >= 0, 1.0, -1.0)

In [ ]:
import time

rows = []
for kernel in ["linear", "rbf"]:
    for C in [0.1, 1.0, 10.0]:
        for pos, neg in [(1, 0), (2, 0), (2, 1)]:
            Xb, yb = binary_subset(Xtr, ytr, pos, neg)
            t0 = time.perf_counter()
            mine = BinarySVM(C=C, kernel=kernel, gamma=GAMMA).fit(Xb, yb)
            t_fit = time.perf_counter() - t0
            sk = SVC(C=C, kernel=kernel, gamma=GAMMA, tol=1e-6).fit(Xb, yb)
            alpha_sk = np.zeros(len(yb))
            alpha_sk[sk.support_] = np.abs(sk.dual_coef_[0])
            Xq, _ = binary_subset(Xte, yte, pos, neg)
            rows.append(
                {
                    "kernel": kernel, "C": C, "pair": f"{pos} vs {neg}",
                    "SMO passes": mine.n_iter_, "fit s": round(t_fit, 2),
                    "#SV mine": len(mine.support_), "#SV sklearn": len(sk.support_),
                    "dual W mine": mine.dual_objective(), "dual W sklearn": mine.dual_objective(alpha_sk),
                    "max |f diff| (test)": np.abs(mine.decision_function(Xq) - sk.decision_function(Xq)).max(),
                    "same predictions": np.mean(mine.predict(Xq) == sk.predict(Xq)),
                }
            )
smo_tbl = pd.DataFrame(rows)
display(smo_tbl.round(5))
rel = (smo_tbl["dual W sklearn"] - smo_tbl["dual W mine"]).abs() / smo_tbl["dual W sklearn"]
print(f"✅ 對偶目標值的相對差：最大 {rel.max():.1e}（應 < 1e-3）")
print(f"✅ 決策值最大差異：{smo_tbl['max |f diff| (test)'].max():.4f}（應 < 0.05）；預測全部一致：{(smo_tbl['same predictions'] == 1).all()}")
print("✅ 支持向量個數相同：", (smo_tbl["#SV mine"] == smo_tbl["#SV sklearn"]).all())

# KKT 檢查：用 rbf、C = 1、類別 2 vs 1
Xb, yb = binary_subset(Xtr, ytr, 2, 1)
kkt_model = BinarySVM(C=1.0, kernel="rbf", gamma=GAMMA).fit(Xb, yb)
yf = yb * kkt_model.decision_function(Xb)
a = kkt_model.alpha_
groups = np.select([a <= 1e-6, a >= kkt_model.C - 1e-6], ["α = 0", "α = C"], "0 < α < C")
display(pd.DataFrame({"group": groups, "y·f(x)": yf}).groupby("group")["y·f(x)"].agg(["size", "min", "max"]).round(4))

In [ ]:
# TODO 9（問答）對偶、SMO、支持向量
#
# (1) 讀 KKT 檢查的表：三組的 y·f(x) 範圍，跟 3C 的 KKT 表一致嗎？（容許 tol 左右的誤差）
#     α = C 的點有沒有 y·f(x) < 0 的？那代表什麼？
# 答：
#
# (2) 讀 SMO 對照表：
#     (a) linear kernel 時，C = 1 和 C = 10 的對偶目標值、支持向量個數一樣嗎？為什麼？
#         （提示：如果兩類本來就能用超平面完全分開，C 大到某個程度之後，還有點需要付 hinge 的錢嗎？—— 那就是 hard margin）
#     (b) rbf 時，C 越大支持向量越多還是越少？用 C 的意義（越界的代價）解釋。
#     (c) 手寫版的決策值和 sklearn 差一點點（不是 1e-12）。為什麼這裡不能期待完全一樣？（提示：tol、停止條件、迭代法 vs 閉式解）
#         那為什麼「對偶目標值幾乎一樣」就足以說明兩者找到同一個解？（提示：凸問題）
# 答：
#
# (3) 決策函數 f(x) = Σ α_i y_i K(x_i, x) + b：
#     (a) 預測時只需要哪些訓練點？跟第四章 KNN 的「要存全部訓練資料」比呢？
#     (b) 支持向量越多，預測越慢還是越快？
#     (c) 如果把一個 α = 0 的訓練點刪掉，重新訓練，答案會變嗎？刪掉一個支持向量呢？
# 答：

### 3D. 多類別：one-vs-one

SVM 天生是二元的。sklearn 的 `SVC` 用 **one-vs-one（OvO）**：$K$ 類兩兩各訓練一個二元 SVM，共 $K(K-1)/2$ 個（Wine：3 個）；預測時每個二元 SVM 投一票給它判定的類別，得票最多的勝出。

- 每個二元 SVM 只用兩類的資料 → 子問題小，SMO 很快（SVM 的訓練成本約 $O(n^2)$ 以上，拆小比較划算）。
- 平手（三類各 1 票）時，sklearn 選「決策值加總」最大的類別；這裡為了簡單，平手時選編號最小的類別（Wine 上幾乎不會平手）。

另一種做法是 **one-vs-rest（OvR）**：$K$ 個二元 SVM，每個「一類 vs 其他全部」，選決策值最大的。`LinearSVC` 用 OvR。

#### 符號表（3D）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| pairs | 所有類別對 $(a, b)$，$a < b$ | 3 對 | `self.pairs_` |
| 子模型 | 第 $p$ 對的 `BinarySVM`；**$a$ → +1、$b$ → −1** | — | `self.models_[p]` |
| winner | 第 $p$ 對對每個查詢點判定的**類別編號** | $(n_\text{query}, \text{\#pairs})$ | `winners` |
| votes | 每個查詢點、每一類的得票數 | $(n_\text{query}, K)$ | `votes` |

**下標公式**（TODO 10）：$\text{votes}_{qc} = \sum_p [\text{winner}_{qp} = c]$ —— 跟第四章 TODO 7 的投票一樣：用 `np.add.at` 或 one-hot 加總。

#### 最小例子：TODO 10（OvO 投票）

2 個查詢點、3 對類別 (0,1)、(0,2)、(1,2)。每個子模型的決策值：`F = [[0.8, 1.2, -0.3], [-0.5, -0.2, -1.0]]`（正 → 該對的第一類贏）。

1. **正確答案**：迴圈逐對投票。目標：第 0 點 votes = (2, 0, 1) → 類別 0；第 1 點 (0, 1, 2) → 類別 2。
2. **積木**：`np.where(F >= 0, a, b)` 得到 winners；`np.add.at`（第四章前置複習 11）。
3. **留給你**：把 winners 加總成 votes。

In [ ]:
# 最小例子：TODO 10（OvO 投票）
pairs_mini = [(0, 1), (0, 2), (1, 2)]
F_mini = np.array([[0.8, 1.2, -0.3], [-0.5, -0.2, -1.0]])  # (n_query=2, #pairs=3)

# ---- 1. 正確答案：迴圈 ----
votes_loop = np.zeros((2, 3), dtype=int)
for q in range(2):
    for p, (a_cls, b_cls) in enumerate(pairs_mini):
        votes_loop[q, a_cls if F_mini[q, p] >= 0 else b_cls] += 1
print("正確答案 votes =\n", votes_loop, "\n預測 =", votes_loop.argmax(axis=1))

# ---- 2. 積木 ----
A_cls = np.array([a for a, _ in pairs_mini])
B_cls = np.array([b for _, b in pairs_mini])
winners_mini = np.where(F_mini >= 0, A_cls[None, :], B_cls[None, :])
print("\nwinners = np.where(F >= 0, a, b)", winners_mini.shape, "\n", winners_mini)
print("rows = np.arange(n_query)[:, None]", np.arange(2)[:, None].shape, "← 跟 (n_query, #pairs) 的 winners 廣播")

# ---- 3. 留給你 ----
# 問題：votes = np.zeros((n_query, K))；np.add.at(votes, (?, ?), 1)
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(OneVsOneSVM.votes_from_decisions(F_mini, pairs_mini, 3))

In [ ]:
from itertools import combinations


class OneVsOneSVM:
    """多類別 SVM：每一對類別一個 BinarySVM，投票。"""

    def __init__(self, **svm_params) -> None:
        self.svm_params = svm_params

    def fit(self, X: ArrayLike, y: ArrayLike) -> Self:
        X, y = np.asarray(X, dtype=float), np.asarray(y)
        self.classes_ = np.unique(y)
        self.pairs_ = list(combinations(range(len(self.classes_)), 2))  # 類別「編號」的配對
        self.models_ = []
        for a, b in self.pairs_:
            Xb, yb = binary_subset(X, y, pos=self.classes_[a], neg=self.classes_[b])  # a → +1、b → −1
            self.models_.append(BinarySVM(**self.svm_params).fit(Xb, yb))
        return self

    def decisions(self, X: ArrayLike) -> np.ndarray:
        """(n_query, #pairs)：每個子模型的決策值。"""
        return np.column_stack([m.decision_function(X) for m in self.models_])

    @staticmethod
    def votes_from_decisions(F: np.ndarray, pairs: list[tuple[int, int]], K: int) -> np.ndarray:
        """F (n_query, #pairs) → votes (n_query, K)：F ≥ 0 投給 pair 的第一類，否則投給第二類。"""
        A = np.array([a for a, _ in pairs])
        B = np.array([b for _, b in pairs])
        winners = np.where(F >= 0, A[None, :], B[None, :])  # (n_query, #pairs) 的類別編號
        votes = np.zeros((F.shape[0], K))
        # TODO 10（程式）：把 winners 加總成 votes
        #   - np.add.at(votes, (列編號, winners), 1)；列編號用 np.arange(n_query)[:, None] 才能跟 winners 廣播
        #   - 不能寫 votes[rows, winners] += 1：同一列重複的位置只會加到一次（第四章 TODO 7）
        raise NotImplementedError("TODO 10：OvO 投票")
        return votes

    def predict(self, X: ArrayLike) -> np.ndarray:
        votes = self.votes_from_decisions(self.decisions(X), self.pairs_, len(self.classes_))
        return self.classes_[votes.argmax(axis=1)]


for kernel, C in [("linear", 1.0), ("rbf", 1.0)]:
    ovo = OneVsOneSVM(C=C, kernel=kernel, gamma=GAMMA).fit(Xtr, ytr)
    sk = SVC(C=C, kernel=kernel, gamma=GAMMA).fit(Xtr, ytr)
    print(
        f"[{kernel}, C={C}] ✅ 與 sklearn SVC 的測試集預測一致比例：{np.mean(ovo.predict(Xte) == sk.predict(Xte)):.4f}（應 = 1.0）",
        f"；訓練集：{np.mean(ovo.predict(Xtr) == sk.predict(Xtr)):.4f}",
        f"；手寫 test accuracy {np.mean(ovo.predict(Xte) == yte):.4f}",
    )
print("每對的支持向量數（rbf）：", {f"{a}v{b}": len(m.support_) for (a, b), m in zip(ovo.pairs_, ovo.models_)}, "；sklearn n_support_（每類）：", sk.n_support_)

### 3E. 實驗：$C$、$\gamma$、尺度、hinge vs log loss

#### 本節名詞

| 名詞 | 定義 |
| --- | --- |
| 間隔內／越界的點 | $y_i f(x_i) < 1$ |
| 局部（local） | RBF 的 $\gamma$ 大時，每個支持向量只影響身邊很小的範圍，邊界可以繞著個別點彎 |
| hinge loss | $\max(0, 1 - y f)$ |
| log loss（$\pm 1$ 寫法） | $\log(1 + e^{-y f})$；第三章的 log loss 換成 $\pm 1$ 標籤後的樣子 |
| 稀疏（sparse）解 | 大部分 $\alpha_i = 0$；只有少數點決定模型 |

這一節直接用 sklearn 的 `SVC`（跟手寫版同一個模型，比較快），畫圖的都是二維子問題。

In [ ]:
# 實驗 1：C。分不開的一對（類別 0 vs 1），linear kernel
X01, y01 = binary_subset(sc2.transform(X_train[FEAT2]), y_train, pos=1, neg=0)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
rows = []
for ax, C in zip(axes, [0.01, 1.0, 100.0]):
    m = SVC(kernel="linear", C=C).fit(X01, y01)
    sv = np.zeros(len(y01), dtype=bool)
    sv[m.support_] = True
    plot_svm_2d(ax, m.decision_function, X01, y01, f"linear, C = {C}", sv=sv)
    yf = y01 * m.decision_function(X01)
    rows.append({"C": C, "margin width 2/‖w‖": 2 / np.linalg.norm(m.coef_), "#SV": len(m.support_),
                 "train errors (y·f < 0)": int((yf < 0).sum()), "inside margin (0 ≤ y·f < 1)": int(((yf >= 0) & (yf < 1)).sum()),
                 "CV accuracy": cross_val_score(SVC(kernel="linear", C=C), X01, y01, cv=cv_single).mean()})
axes[0].legend(loc="upper left", fontsize=8)
plt.show()
display(pd.DataFrame(rows).set_index("C").round(3))

In [ ]:
# 實驗 2：γ。三類、二維（flavanoids × color_intensity），RBF kernel，C = 1
X3 = sc2.transform(X_train[FEAT2])
xx, yy = np.meshgrid(np.linspace(X3[:, 0].min() - 0.5, X3[:, 0].max() + 0.5, 300), np.linspace(X3[:, 1].min() - 0.5, X3[:, 1].max() + 0.5, 300))
gammas = [0.1, 1.0, 10.0, 100.0]
fig, axes = plt.subplots(1, 4, figsize=(19, 4.3))
rows = []
for ax, g in zip(axes, gammas):
    m = SVC(kernel="rbf", C=1.0, gamma=g).fit(X3, ytr)
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5, 2.5], colors=["#cfe0f1", "#f8d9c4", "#d5ecd0"], alpha=0.8)
    for k in range(3):
        ax.scatter(X3[ytr == k, 0], X3[ytr == k, 1], s=12, label=f"class {k}")
    ax.set_title(f"RBF, gamma = {g}")
    rows.append({"gamma": g, "radius ~ 1/sqrt(gamma)": 1 / np.sqrt(g), "#SV": int(m.n_support_.sum()), "train acc": m.score(X3, ytr),
                 "CV acc": cross_val_score(SVC(kernel="rbf", C=1.0, gamma=g), X3, ytr, cv=cv_single).mean()})
axes[0].legend(fontsize=8)
plt.show()
display(pd.DataFrame(rows).set_index("gamma").round(3))

In [ ]:
# TODO 11（問答）C 與 γ
#
# (1) 實驗 1（分不開的 0 vs 1）：
#     (a) C 從 0.01 到 100，間隔寬度、支持向量數、間隔內的點數各怎麼變？
#     (b) C = 0.01 時幾乎所有點都是支持向量。用 3C 的 KKT 表解釋：間隔很寬時，大部分點的 y·f 是多少、α 是多少？
#     (c) 哪個 C 的 CV 最好？C 越大一定越好嗎？跟第三章「C 越大 = 正則化越弱」是同一個方向嗎？
# 答：
#
# (2) 實驗 2（RBF 的 γ）：
#     (a) γ = 100 的圖長什麼樣子？train acc 與 CV acc 各多少？這是過擬合還是欠擬合？
#     (b) γ 大 → 「半徑」1/√γ 小 → 每個支持向量只影響身邊。對照第四章 KNN：γ 大的 RBF SVM 像 k 大還是 k 小的 KNN？
#     (c) γ 很小時 RBF kernel 會變得「幾乎每對點都差不多相似」。這時 RBF SVM 的邊界接近什麼形狀？
#         （提示：exp(−γ‖x−z‖²) ≈ 1 − γ‖x−z‖²，展開後跟 x·z 有關）
# 答：
#
# (3) C 和 γ 都控制複雜度，但方式不同：一個是「多在乎分錯」，一個是「邊界能多彎」。
#     有沒有可能「C 小、γ 大」和「C 大、γ 小」得到差不多的 CV？（第 5 節的 heatmap 會看到一條斜的帶狀）
# 答：

In [ ]:
# 實驗 3：尺度。標準化 vs 不標準化（全部 13 個特徵、三類）
rows = []
for name, kw in [("linear", dict(kernel="linear", C=1.0)), ("rbf (gamma='scale')", dict(kernel="rbf", C=1.0)), ("rbf (gamma=1/13 fixed)", dict(kernel="rbf", C=1.0, gamma=1 / 13))]:
    rows.append({
        "model": name,
        "raw": cross_val_score(SVC(**kw), X_train, y_train, cv=cv).mean(),
        "standardized": cross_val_score(make_pipeline(StandardScaler(), SVC(**kw)), X_train, y_train, cv=cv).mean(),
    })
display(pd.DataFrame(rows).set_index("model").round(4))

# 不標準化時，RBF kernel 裡的平方距離有多少來自 proline？
D_raw = ((X_train.to_numpy()[:, None, :] - X_train.to_numpy()[None, :, :]) ** 2)  # (n, n, d)：只為了拆成每個特徵的貢獻
share = D_raw.sum(axis=(0, 1)) / D_raw.sum()
print("raw 平方距離中各特徵的占比（前 3）：", pd.Series(share, index=X_train.columns).sort_values(ascending=False).head(3).round(4).to_dict())
K_raw = rbf_kernel_matrix(X_train, X_train, 1 / 13)
print(f"raw + gamma=1/13 的 Gram 矩陣：非對角線元素 > 0.01 的比例 = {(K_raw[~np.eye(len(K_raw), dtype=bool)] > 0.01).mean():.4f}")

In [ ]:
# 實驗 4：hinge vs log loss —— SVM 與邏輯回歸（類別 2 vs 1、二維、C = 1）
from sklearn.linear_model import LogisticRegression

zz = np.linspace(-3, 3, 200)
lr = LogisticRegression(C=1.0).fit(X2, y2)
svm = SVC(kernel="linear", C=1.0).fit(X2, y2)
yf_svm, yf_lr = y2 * svm.decision_function(X2), y2 * lr.decision_function(X2)
grad_svm = (yf_svm < 1).astype(float)  # 每個點對梯度的「權重」：hinge 的斜率絕對值
grad_lr = 1 / (1 + np.exp(yf_lr))  # log loss 的斜率絕對值 σ(−y f)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
axes[0].plot(zz, np.maximum(0, 1 - zz), label="hinge  max(0, 1 − yf)")
axes[0].plot(zz, np.log1p(np.exp(-zz)) / np.log(2), label="log loss / ln 2")
axes[0].plot(zz, (zz < 0).astype(float), ":", label="0-1 loss")
axes[0].set_xlabel("y · f(x)")
axes[0].set_ylabel("loss")
axes[0].legend()
axes[0].set_title("Loss as a function of the margin")
axes[1].scatter(yf_svm, grad_svm, s=12, label="SVM (hinge)")
axes[1].scatter(yf_lr, grad_lr, s=12, label="LogReg (log loss)")
axes[1].set_xlabel("y · f(x) on training data")
axes[1].set_ylabel("|d loss / d(yf)|  (pull on the boundary)")
axes[1].legend()
axes[1].set_title("Which points pull on the boundary?")
plot_svm_2d(axes[2], svm.decision_function, X2, y2, "SVM (solid) vs LogReg (dotted)")
xs = np.linspace(X2[:, 0].min(), X2[:, 0].max(), 50)
axes[2].plot(xs, -(lr.coef_[0, 0] * xs + lr.intercept_[0]) / lr.coef_[0, 1], "m:", lw=2)
axes[2].set_ylim(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5)
fig.tight_layout()
plt.show()
print(f"SVM：拉邊界的點（權重 > 0）{int((grad_svm > 0).sum())} / {len(y2)}；LogReg：權重 > 0.01 的點 {int((grad_lr > 0.01).sum())} / {len(y2)}")
print("SVM w 方向：", (svm.coef_[0] / np.linalg.norm(svm.coef_[0])).round(3), "；LogReg w 方向：", (lr.coef_[0] / np.linalg.norm(lr.coef_[0])).round(3))

In [ ]:
# TODO 12（問答）尺度與 hinge loss
#
# (1) 實驗 3：
#     (a) 不標準化時，linear 與 rbf 各掉了多少？哪一個掉得比較多？
#     (b) raw 的平方距離有多少來自 proline？gamma = 1/13 時 Gram 矩陣幾乎全是 0（除了對角線）—— 這代表每個點跟誰都「不像」，
#         SVM 會退化成什麼？（提示：f(x) = Σ α_i y_i K(x_i, x) + b，K 幾乎都是 0）
#     (c) gamma='scale' 會自動用 1/(d·Var(X)) 調整，為什麼 raw 還是比標準化差？（提示：它只用「所有特徵合起來」的一個變異數）
#     (d) 線性 SVM 也掉了，但理由跟 RBF 不一樣：½‖w‖² 對不同尺度的特徵公平嗎？（對照第三章的 L2）
# 答：
#
# (2) 實驗 4：
#     (a) 左圖：y·f ≥ 1 之後，hinge 和 log loss 各是多少？y·f 很負（嚴重分錯）時，兩者各怎麼長？
#     (b) 中圖：SVM 有幾個點在「拉」邊界？邏輯回歸呢？這就是「SVM 的解是稀疏的」的意思。
#     (c) 右圖：兩條邊界差很多嗎？
#     (d) 邏輯回歸給機率，SVM 只給決策值。如果你需要「這支酒是品種 2 的機率」，用 SVM 要怎麼辦？（sklearn 的 probability=True 做了什麼 —— 見待補強）
# 答：

---
## 4. 套件實作：sklearn `SVC` 與 `LinearSVC`

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `SVC` | libsvm：對偶問題 + SMO（挑 $(i, j)$ 的方法比簡化版聰明）；多類別用 OvO |
| `support_`、`n_support_` | 支持向量的列號、每一類的支持向量數 |
| `dual_coef_` | 支持向量的 $\alpha_i y_i$（多類別時排成 $(K-1, n_\text{SV})$ 的特殊格式） |
| `LinearSVC` | liblinear：直接解原始問題，預設 **squared hinge** $\max(0, 1 - yf)^2$、**截距也被正則化**、多類別用 **OvR** —— 跟 `SVC(kernel="linear")` 不是同一個模型 |
| `decision_function_shape` | `SVC` 預設 `"ovr"`：把 OvO 的票數整理成 $(n, K)$ 的形狀輸出（預測仍是 OvO 投票） |

#### 程式結構圖

```
svm_pipe = Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="rbf", C=1.0, gamma="scale"))])
├── svm_pipe["scaler"]：mean_、scale_
└── svm_pipe["svm"]   ：support_、n_support_、dual_coef_、intercept_、classes_
參數名：svm__C、svm__gamma、svm__kernel
```

In [ ]:
from sklearn.svm import LinearSVC

svm_pipe = Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="rbf", C=1.0, gamma="scale"))]).fit(X_train, y_train)
ovo_rbf = OneVsOneSVM(C=1.0, kernel="rbf", gamma=GAMMA).fit(Xtr, ytr)
print("✅ 手寫 OvO（rbf, C=1）vs sklearn Pipeline：測試集預測一致比例", np.mean(ovo_rbf.predict(Xte) == svm_pipe.predict(X_test)), "（應 = 1.0）")
print("支持向量：每類", svm_pipe["svm"].n_support_, "，共", svm_pipe["svm"].n_support_.sum(), "/", len(y_train))
print("dual_coef_ shape：", svm_pipe["svm"].dual_coef_.shape)

variants = {
    "SVC linear C=1": Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="linear", C=1.0))]),
    "LinearSVC C=1 (squared hinge, OvR)": Pipeline([("scaler", StandardScaler()), ("svm", LinearSVC(C=1.0))]),
    "SVC rbf C=1 (default)": svm_pipe,
    "SVC poly deg=3 C=1": Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="poly", degree=3, C=1.0))]),
}
for name, m in variants.items():
    baseline.loc[name] = cv_summary(m)
display(baseline)

lin_svc = variants["SVC linear C=1"].fit(X_train, y_train)["svm"]
lin_l = variants["LinearSVC C=1 (squared hinge, OvR)"].fit(X_train, y_train)["svm"]
print("SVC(linear) coef_ shape：", lin_svc.coef_.shape, "（OvO：每一對一列）；LinearSVC coef_ shape：", lin_l.coef_.shape, "（OvR：每一類一列）")

In [ ]:
# TODO 13（問答）
#
# (1) 四種變體的 CV accuracy 差多少？跟 std 比呢？在這份資料上，非線性 kernel 有必要嗎？
#     （提示：TODO 1 (2)、3C 的 linear 對照表 —— 兩兩之間在 13 維裡是不是本來就線性可分？）
# 答：
#
# (2) SVC(kernel="linear") 與 LinearSVC：
#     (a) coef_ 的 shape 為什麼不同？
#     (b) LinearSVC 的三個差異（squared hinge、截距被正則化、OvR）各會讓結果往哪裡偏？
#     (c) 什麼時候該用 LinearSVC？（提示：SVC 的 Gram 矩陣是 n×n；文字分類 n = 100 萬、d = 10 萬時呢？）
# 答：
#
# (3) 支持向量佔訓練資料的比例是多少？（看 n_support_ 的總和）
#     如果比例很高（例如 > 50%），SVM「稀疏」的優點還在嗎？這通常代表 C、γ 怎麼了？
# 答：

---
## 5. 調參

全程只用 `X_train`（測試集鎖住），用重複 5-fold CV（每組 50 個分數）。

| 類別 | 超參數 | 說明 |
| --- | --- | --- |
| 正則化 | `svm__C` | 越大越不容忍錯誤 = 越複雜；尺度型 → log 尺度 |
| 容量 | `svm__gamma`（RBF） | 越大越局部 = 越複雜；尺度型 → log 尺度 |
| 容量 | `svm__kernel` | linear／rbf／poly |
| 前處理 | scaler | 一定要（實驗 3），不調 |

**選模型的指標用 accuracy**。

#### 本節名詞

- **C × γ heatmap**：RBF SVM 調參的標準畫法；好的區域通常是一條**斜的帶狀**（TODO 11 (3)）。
- **2D 的 1-SE**：門檻 = 最佳的 mean − 最佳的 std；在門檻內挑「最簡單」的。RBF SVM 的「簡單」：**γ 小優先**（邊界比較平滑），γ 相同再挑 **C 小**。

In [ ]:
from sklearn.model_selection import GridSearchCV, learning_curve, validation_curve

# 單一參數掃描：linear 的 C、rbf 的 gamma（C = 1）
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
vc_tbls = {}
for ax, (pipe, param, values, title) in zip(
    axes,
    [
        (Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="linear"))]), "svm__C", np.logspace(-3, 3, 13), "linear: C (larger = more complex)"),
        (svm_pipe, "svm__gamma", np.logspace(-4, 2, 13), "rbf, C=1: gamma (larger = more complex)"),
    ],
):
    tr_s, va_s = validation_curve(pipe, X_train, y_train, param_name=param, param_range=values, cv=cv, scoring="accuracy", n_jobs=-1)
    for name, s in [("train", tr_s), ("validation", va_s)]:
        ax.plot(values, s.mean(1), "o-", ms=3, label=name)
        ax.fill_between(values, s.mean(1) - s.std(1), s.mean(1) + s.std(1), alpha=0.2)
    ax.set_xscale("log")
    ax.set_xlabel(param.split("__")[1])
    ax.set_ylabel("accuracy")
    ax.set_title(title)
    ax.legend()
    vc_tbls[param] = pd.DataFrame({"train": tr_s.mean(1), "val": va_s.mean(1), "val_std": va_s.std(1)}, index=values)
plt.show()
for param, tbl in vc_tbls.items():
    print(param)
    display(tbl.round(4).T)

In [ ]:
# 2D 搜尋：C × gamma（RBF）
param_grid = {"svm__C": np.logspace(-2, 3, 11), "svm__gamma": np.logspace(-4, 1, 11)}
gs = GridSearchCV(svm_pipe, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
gs.fit(X_train, y_train)
res = pd.DataFrame(
    {
        "C": gs.cv_results_["param_svm__C"].astype(float),
        "gamma": gs.cv_results_["param_svm__gamma"].astype(float),
        "mean": gs.cv_results_["mean_test_score"],
        "std": gs.cv_results_["std_test_score"],
    }
)
print("best_params_：", {k: round(float(v), 5) for k, v in gs.best_params_.items()}, "  CV accuracy：", round(gs.best_score_, 4),
      " 支持向量數：", gs.best_estimator_["svm"].n_support_.sum())

heat = res.pivot_table(index="gamma", columns="C", values="mean")
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="viridis", ax=ax, annot_kws={"size": 7},
            xticklabels=[f"{c:.2g}" for c in heat.columns], yticklabels=[f"{g:.2g}" for g in heat.index])
ax.set_title("CV accuracy (RBF SVM): rows = gamma, columns = C")
plt.show()

#### 最小例子：TODO 14（二維的 1-SE）

6 列的假結果表，accuracy 越高越好；「簡單」= γ 小優先，γ 相同時 C 小優先。

1. **正確答案**：逐列判斷，再用 `min(..., key=(gamma, C))` 挑。目標：`gamma = 0.01、C = 10`。
2. **積木**：門檻、布林遮罩、`sort_values` 多個欄位。
3. **留給你**：`sort_values([...], ascending=[...]).iloc[0]`。

In [ ]:
# 最小例子：TODO 14（二維的 1-SE）
res_mini = pd.DataFrame(
    {
        "C": [1.0, 10.0, 100.0, 1.0, 10.0, 0.1],
        "gamma": [0.1, 0.01, 0.001, 0.01, 0.1, 0.001],
        "mean": [0.990, 0.982, 0.960, 0.970, 0.985, 0.700],
        "std": [0.010, 0.012, 0.020, 0.020, 0.015, 0.050],
    }
)
display(res_mini)

# ---- 1. 正確答案：逐列判斷 ----
best_loop = max(res_mini.to_dict("records"), key=lambda r: r["mean"])
thr_loop = best_loop["mean"] - best_loop["std"]
ok_loop = [r for r in res_mini.to_dict("records") if r["mean"] >= thr_loop]
for r in res_mini.to_dict("records"):
    print(f"C={r['C']:<6} gamma={r['gamma']:<6} acc={r['mean']:.3f} ≥ {thr_loop:.3f}？{r['mean'] >= thr_loop}")
print("正確答案：", min(ok_loop, key=lambda r: (r["gamma"], r["C"])))

# ---- 2. pandas 的積木 ----
best_mini = res_mini.loc[res_mini["mean"].idxmax()]
ok_mini = res_mini[res_mini["mean"] >= best_mini["mean"] - best_mini["std"]]
display(ok_mini)

# ---- 3. 留給你 ----
# 問題：ok_mini.sort_values(["gamma", "C"], ascending=[?, ?]).iloc[0] 怎麼填？

In [ ]:
# 二維的 1-SE：跟 CV 最佳差不到一個標準差的 (C, gamma) 中，挑最簡單的
best = res.loc[res["mean"].idxmax()]

# TODO 14（程式）
#   - ok：mean ≥ best["mean"] − best["std"] 的所有列
#   - choice：ok 裡「gamma 最小」的；gamma 相同時「C 最小」的
raise NotImplementedError("TODO 14：二維 1-SE")

print("CV 最佳 ：", best.round(5).to_dict())
print("1-SE 選擇：", choice.round(5).to_dict())
print("✅ 1-SE 的 gamma 應 ≤ CV 最佳的 gamma：", choice["gamma"] <= best["gamma"])

from sklearn.base import clone

model_best = gs.best_estimator_
model_1se = clone(svm_pipe).set_params(svm__C=choice["C"], svm__gamma=choice["gamma"]).fit(X_train, y_train)
print("支持向量數：CV 最佳", model_best["svm"].n_support_.sum(), "；1-SE", model_1se["svm"].n_support_.sum())

# 學習曲線
sizes, tr_s, va_s = learning_curve(model_1se, X_train, y_train, cv=cv_single, scoring="accuracy", train_sizes=np.linspace(0.2, 1.0, 6))
fig, ax = plt.subplots(figsize=(7, 4))
for name, s in [("train", tr_s), ("validation", va_s)]:
    ax.plot(sizes, s.mean(1), "o-", label=name)
    ax.fill_between(sizes, s.mean(1) - s.std(1), s.mean(1) + s.std(1), alpha=0.2)
ax.set_xlabel("training samples")
ax.set_ylabel("accuracy")
ax.set_title("Learning curve (1-SE RBF SVM)")
ax.legend()
plt.show()
display(pd.DataFrame({"n": sizes, "train": tr_s.mean(1), "val": va_s.mean(1), "val_std": va_s.std(1)}).round(4).T)

In [ ]:
# TODO 15（問答）調參
#
# (1) validation curve：
#     (a) linear 的 C：哪一段欠擬合？C 大到某個程度以後，train 與 val 都不再變 —— 為什麼？（TODO 9 (2)(a)）
#     (b) rbf 的 gamma：過擬合在左邊還是右邊？gamma 最大時 train、val 各多少？
# 答：
#
# (2) heatmap：
#     (a) 好的區域（例如 ≥ 0.97）是不是一條斜的帶狀？沿著帶狀，C 變大時 gamma 要變大還是變小才能維持差不多的分數？為什麼？
#     (b) 左上角（C 小、gamma 小）和右下角（C 大、gamma 大）各是欠擬合還是過擬合？
#     (c) 很多格都在 0.98 附近。最佳那一格跟其他高分的格子差多少？跟 std 比呢？
# 答：
#
# (3) 1-SE：
#     (a) 選了哪一組？支持向量數比 CV 最佳多還是少？為什麼「γ 小」的模型支持向量反而可能比較多？（TODO 11 (1)(b)）
#     (b) 這裡「簡單」換到了什麼實際好處？（對照第四章 TODO 16 (2)(d) 的討論）
#     (c) 學習曲線：train 與 val 的差距多大？再多給資料會有幫助嗎？
# 答：

---
## 6. 測試集評估

模型都已經用 CV 決定好，測試集只評估一次。**測試集只有 36 筆（1 筆 ≈ 2.8%）**，排名請搭配 CV 一起看。對照組都用標準化後的資料、沒有調參：

- **邏輯回歸**（第三章，$C = 1$）、**KNN**（第四章，$k = 5$）、**決策樹**（第五章，深度 3）、**Gaussian NB**（第六章）。

#### 指標

- **accuracy**、**macro precision／recall／F1**（多類別時每類各算一次再平均）。
- `log_result`／`compare`：成績寫進 `results/wine.csv`，依 accuracy 排序。SVM 沒有 `predict_proba`（除非 `probability=True`），所以不算 ROC AUC。

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from learning_map.metrics import classification_metrics
from learning_map.results import compare, log_result

BENCH = "wine"
ovo_lin = OneVsOneSVM(C=1.0, kernel="linear").fit(Xtr, ytr)
refs = {
    "LogReg C=1 (sklearn, ref)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "KNN k=5 (sklearn, ref)": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "Tree depth 3 (sklearn, ref)": DecisionTreeClassifier(max_depth=3, random_state=0),
    "Gaussian NB (sklearn, ref)": GaussianNB(),
}
for m in refs.values():
    m.fit(X_train, y_train)

# 名稱: (預測, 要記下的參數)
eval_models = {
    "Dummy prior": (DummyClassifier(strategy="prior").fit(X_train, y_train).predict(X_test), {}),
    "NearestCentroid (std)": (make_pipeline(StandardScaler(), NearestCentroid()).fit(X_train, y_train).predict(X_test), {}),
    "SVM linear C=1 OvO (scratch SMO)": (ovo_lin.predict(Xte), {"kernel": "linear", "C": 1.0}),
    "SVM rbf C=1 OvO (scratch SMO)": (ovo_rbf.predict(Xte), {"kernel": "rbf", "C": 1.0, "gamma": round(GAMMA, 4)}),
    "SVC rbf default (sklearn)": (svm_pipe.predict(X_test), {"C": 1.0, "gamma": "scale"}),
    "SVC rbf CV-best (sklearn)": (model_best.predict(X_test), {"C": round(float(best["C"]), 4), "gamma": round(float(best["gamma"]), 5)}),
    "SVC rbf 1-SE (sklearn)": (model_1se.predict(X_test), {"C": round(float(choice["C"]), 4), "gamma": round(float(choice["gamma"]), 5)}),
    "LinearSVC C=1 (sklearn)": (variants["LinearSVC C=1 (squared hinge, OvR)"].predict(X_test), {"C": 1.0}),
    **{name: (m.predict(X_test), {}) for name, m in refs.items()},
}

test_scores = {}
for name, (pred, params) in eval_models.items():
    m = classification_metrics(y_test, pred)
    test_scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(test_scores).T)
compare(BENCH, "accuracy")

---
## 7. 診斷

#### 本節名詞

- **混淆矩陣**：列 = 真實品種、欄 = 預測品種。
- **決策值的「信心」**：OvO 的每個子模型給一個決策值 $f$；$|f|$ 越大離邊界越遠。這裡用「贏的那一類在它參與的兩個子模型中，最小的 $|f|$」當作這一筆的信心 —— 最弱的那一環。
- **測試集 vs 重複 CV**：36 筆的測試集只是「一次抽樣」；50 個 CV 分數的分布能告訴我們這一次的成績是不是運氣。

In [ ]:
from learning_map.metrics import plot_confusion_matrix

final = model_1se
pred_te = final.predict(X_test)
plot_confusion_matrix(y_test, pred_te)
plt.title("1-SE RBF SVM on test")
plt.show()
print(f"測試集錯 {int((pred_te != yte).sum())} 筆 / {len(yte)}")

# 每一筆的「最弱的一環」：用手寫 OvO 拿每個子模型的決策值（同樣的 C、gamma）
ovo_final = OneVsOneSVM(C=float(choice["C"]), kernel="rbf", gamma=float(choice["gamma"])).fit(Xtr, ytr)
F_te = ovo_final.decisions(Xte)  # (n_test, 3)
pred_ovo = ovo_final.predict(Xte)
weakest = np.array(
    [min(abs(F_te[q, p]) for p, (a, b) in enumerate(ovo_final.pairs_) if pred_ovo[q] in (a, b)) for q in range(len(yte))]
)
diag = pd.DataFrame({"true": yte, "pred": pred_ovo, "weakest |f|": weakest}, index=X_test.index)
display(diag.sort_values("weakest |f|").head(8).round(3))
print("答對的 weakest |f| 中位數：", round(float(np.median(weakest[pred_ovo == yte])), 3),
      "；答錯的：", np.round(weakest[pred_ovo != yte], 3) if (pred_ovo != yte).any() else "（沒有答錯）")

# 測試集成績 vs 重複 CV 的分布
cv_scores = cross_val_score(final, X_train, y_train, cv=cv)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(cv_scores, bins=np.arange(0.8, 1.02, 1 / 29), alpha=0.7, label="50 CV fold scores (~29 samples each)")
ax.axvline(np.mean(pred_te == yte), color="r", ls="--", label=f"test accuracy ({len(yte)} samples)")
ax.set_xlabel("accuracy")
ax.legend()
plt.show()
print(f"CV：平均 {cv_scores.mean():.4f}、最差 fold {cv_scores.min():.3f}、{(cv_scores < 1).mean():.0%} 的 fold 至少錯 1 筆")

In [ ]:
# TODO 16（問答，本章總結）
#
# (1) 錯在哪裡：
#     (a) 測試集錯幾筆？錯在哪兩類之間？跟 EDA 的散點圖（哪兩類比較擠）一致嗎？
#     (b) weakest |f| 最小的那幾筆，是不是剛好就是錯的、或差點錯的？「離邊界近」可以當作不確定的訊號嗎？
#         （如果測試集剛好全對，就看 weakest |f| 最小的幾筆：它們是哪兩類之間的？）
# 答：
#
# (2) 測試集 vs CV：
#     (a) 測試集 accuracy 落在 CV 分布的哪裡？
#     (b) 第 6 節的表裡，幾個模型的測試 accuracy 差 1–2 筆。你能說哪一個「真的比較好」嗎？要怎麼比較才公平？
#         （提示：用同一組 RepeatedStratifiedKFold，比較兩個模型「每個 fold」的分數差 —— 配對比較）
# 答：
#
# (3) 跟前幾章比（看第 6 節與第 4 節的表）：
#     (a) 比較表（填空）：
#
#         |                        | SVM (RBF) | 邏輯回歸 | KNN | 決策樹 | 樸素貝氏 |
#         | 需要標準化嗎           |           |          |     |        |          |
#         | 決策邊界的形狀         |           |          |     |        |          |
#         | 哪些訓練點決定模型     |           |          |     |        |          |
#         | 給不給機率             |           |          |     |        |          |
#         | 訓練成本（n 很大時）   |           |          |     |        |          |
#         | 主要超參數             |           |          |     |        |          |
#
#     (b) 什麼情況下選 SVM？什麼情況下不會選？（提示：n 的大小、特徵數、需不需要機率、可解釋性）
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

Wine（train / test 依載入結果；CV = 5-fold 分層 × 10 次；測試集 1 筆 ≈ 2.8%）。

| 方法 | 做法 | 關鍵性質 | CV accuracy | test accuracy |
| --- | --- | --- | --- | --- |
| Dummy prior | 猜最多的類 | | | |
| NearestCentroid（標準化） | 每類一個中心 | | | |
| 線性 SVM（手寫 primal） | hinge + 次梯度（二元、二維） | | — | — |
| SVM linear OvO（手寫 SMO） | 對偶 + 簡化版 SMO | | | |
| SVM rbf OvO（手寫 SMO） | RBF kernel | | | |
| SVC rbf CV 最佳 | C × γ grid | | | |
| SVC rbf 1-SE | γ 小、C 小優先 | | | |
| 邏輯回歸／KNN／決策樹／NB（對照） | | | — | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ 從 primal 推到 dual：拉格朗日函數、KKT 條件的完整推導；為什麼資料只以內積出現
- ☐ Mercer 定理：什麼函數可以當 kernel（Gram 矩陣半正定）；RBF 對應的無限維特徵空間
- ☐ 間隔與泛化：VC 維度與 margin bound；為什麼「間隔大」理論上泛化比較好
- ☐ hinge loss 是 0-1 loss 的凸上界；各種 surrogate loss 的比較

**演算法**
- ☐ 完整的 SMO：Platt 的 heuristics、libsvm 的 working set selection（WSS3）、shrinking、kernel cache
- ☐ Pegasos（隨機次梯度）、坐標下降（liblinear 的 dual CD）
- ☐ 大資料的 kernel 近似：Nyström、Random Fourier Features（`sklearn.kernel_approximation`）

**延伸模型**
- ☐ 機率輸出：Platt scaling（`probability=True` 內部的 5-fold CV + sigmoid）
- ☐ SVR（ε-insensitive loss）、One-Class SVM（異常偵測，第八章 Isolation Forest 對照）
- ☐ 多類別的其他做法：Crammer–Singer 多類別 SVM、OvR vs OvO 的取捨
- ☐ 類別不平衡：`class_weight`

**評估**
- ☐ 配對比較兩個模型：corrected repeated k-fold t-test（Nadeau & Bengio）、McNemar
- ☐ nested CV：調參與評估分開的完整流程（小資料時特別重要）